In [ ]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
API_KEY = "8MDAU8GFRVWCC76P"
symbol = "AAPL"

In [ ]:
url = "https://www.alphavantage.co/query"

In [ ]:
params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": symbol,
    "apikey": API_KEY
}

In [ ]:
response = requests.get(url, params=params)

In [ ]:
response.status_code

In [ ]:
data = response.json()

In [ ]:
type(data)

In [ ]:
data.keys()

In [ ]:
data.values()

In [ ]:
data.items()

In [ ]:
data["Meta Data"]

In [ ]:
data["Time Series (Daily)"]

In [ ]:
data["Time Series (Daily)"]

In [ ]:
stock_data = data["Time Series (Daily)"]

In [ ]:
df = pd.DataFrame.from_dict(stock_data, orient="index")

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.columns = ["Open", "High", "Low", "Close", "Volume"]

In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
df.index

In [ ]:
df.index = pd.to_datetime(df.index)

In [ ]:
df.index

In [ ]:
df = df.sort_index()

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(df.index, df['Close'])

plt.title('Apple (AAPL) Closing Price')
plt.xlabel('Date')
plt.ylabel('Closing Price')
plt.xticks(rotation=45)
plt.grid(True)

plt.show()

In [ ]:
data = df[['Close']].copy()

data.head()

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))

scaled_data = scaler.fit_transform(data)

print("Original shape:", data.shape)
print("Scaled shape:", scaled_data.shape)

print("\nFirst 5 original values:")
print(data.head())

print("\nFirst 5 scaled values:")
print(scaled_data[:5])


Create Time-Series Sequences

In [ ]:
scaled_data

In [ ]:
scaled_data[10-10:10,0]

In [ ]:
TIME_STEPS = 10

X = []
y = []

for i in range(TIME_STEPS, len(scaled_data)):

    X.append(scaled_data[i-TIME_STEPS:i, 0])
    y.append(scaled_data[i, 0])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

Reshape X for RNN

In [ ]:
X = X.reshape(X.shape[0], X.shape[1], 1)

print("X shape:", X.shape)
print("y shape:", y.shape)

Train/Test Split for Time Series

In [ ]:
train_size = int(len(X) * 0.8)

X_train = X[:train_size]
X_test = X[train_size:]

y_train = y[:train_size]
y_test = y[train_size:]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test :", X_test.shape)
print("y_test :", y_test.shape)

Build the RNN Model

In [ ]:
model = Sequential([SimpleRNN(50,activation='tanh',input_shape=(X_train.shape[1],X_train.shape[2]) ),Dense(1)])

Compile the Model

In [ ]:
model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

In [ ]:
model.summary()

Train the RNN

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=8,
    validation_data=(X_test, y_test),
    verbose=1
)

In [ ]:


plt.figure(figsize=(12, 5))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title('RNN Training vs Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
model.summary()

Make Prediction

In [ ]:
y_pred = model.predict(X_test)

print("Prediction shape:", y_pred.shape)

In [ ]:
print(y_pred[:5])

Convert Predictions Back to Actual Prices

In [ ]:
y_pred_actual = scaler.inverse_transform(y_pred)

In [ ]:
y_test_actual = scaler.inverse_transform(
    y_test.reshape(-1, 1)
)

In [ ]:
print("Actual prices:")
print(y_test_actual[:5])

print("\nPredicted prices:")
print(y_pred_actual[:5])

In [ ]:
#Create Actual vs Predicted Graph

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    y_test_actual,
    label='Actual Close Price'
)

plt.plot(
    y_pred_actual,
    label='Predicted Close Price'
)

plt.title('Apple Stock Price: Actual vs Predicted')
plt.xlabel('Test Trading Days')
plt.ylabel('Close Price')
plt.legend()
plt.grid(True)

plt.show()

Calculate Evaluation Metrics

In [ ]:


mae = mean_absolute_error(y_test_actual, y_pred_actual)

rmse = np.sqrt(
    mean_squared_error(y_test_actual, y_pred_actual)
)

r2 = r2_score(y_test_actual, y_pred_actual)

print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

In [ ]:
import joblib

joblib.dump(scalar,"apple_scaler.pkl")

In [ ]:
model.save("apple_model.keras")

In [ ]:
from tensorflow.keras.models import load_model
import joblib

model = load_model("apple_model.keras")
scaler = joblib.load("apple_scaler.pkl")

In [ ]:
def predict_price(X):
    prediction = model.predict(X, verbose=0)
    prediction = scaler.inverse_transform(prediction)

    return prediction[0][0]

In [ ]:
price = predict_price(x_test[-1].reshape(1, 10, 1))

print("Predicted Price:", price)

In [ ]:
def predict_by_date(date):
    date = pd.to_datetime(date)

    data_before = df[df.index < date]


    prices = data_before["Close"].values[-10:]
    if len(prices) < 10:
        return "Not enough data"

  
    prices = scaler.transform(prices.reshape(-1, 1))


    X = prices.reshape(1, 10, 1)

    prediction = model.predict(X, verbose=0)

    price = scaler.inverse_transform(prediction)

    return float(price[0][0])

In [ ]:
price = predict_by_date("2026-11-26")

print(price)